# Optional benchmark: LLM zero-shot / few-shot classification

Этот notebook проверяет, как LLM справляется с задачей бинарной классификации новостных фрагментов без обучения модели на датасете.

Benchmark является optional: он зависит от внешнего API, ключа доступа, стоимости запросов, версии LLM и prompt design. Результаты notebook не используются для обучения или выбора моделей в основном проекте.

Цель benchmark — отдельно оценить zero-shot и few-shot подходы для задачи:

`title + text_fragment + entity_norm -> alert_flag`


## 1. Цель LLM benchmark

LLM benchmark проверяет, может ли большая языковая модель классифицировать банковские новостные фрагменты как риск-сигнал (`1`) или отсутствие значимого риска (`0`) без обучения на текущем датасете.

Проверяются два режима:

- zero-shot: только инструкция и фрагмент;
- few-shot: инструкция, несколько train-примеров и фрагмент.

Результаты используются только как отдельный исследовательский benchmark.


## 2. Загрузка данных

Используется датасет:

```text
homework_04_dataset/data/dataset_for_training.csv
```

Для benchmark используется готовый `test` split. Prompt и параметры запуска фиксируются до просмотра результатов; test не используется для донастройки prompt.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "homework_04_dataset").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "homework_04_dataset" / "data" / "dataset_for_training.csv"

df = pd.read_csv(DATA_PATH)
df["model_text"] = (
    df["title"].fillna("") + " " +
    df["text_fragment"].fillna("") + " " +
    df["entity_norm"].fillna("")
)

train_df = df[df["split"] == "train"].copy()
valid_df = df[df["split"] == "valid"].copy()
test_df = df[df["split"] == "test"].copy()

display(pd.DataFrame({
    "split": ["train", "valid", "test"],
    "n_rows": [len(train_df), len(valid_df), len(test_df)],
    "positive_rate": [
        train_df["alert_flag"].mean(),
        valid_df["alert_flag"].mean(),
        test_df["alert_flag"].mean(),
    ],
}))


,split,n_rows,positive_rate
0,train,587,0.303237
1,valid,126,0.301587
2,test,126,0.301587


Benchmark можно запускать на подвыборке или на полном test. По умолчанию используется полный test, если `RUN_FULL_LLM_BENCHMARK = True`.


In [2]:
RUN_FULL_LLM_BENCHMARK = True
LLM_SAMPLE_SIZE = 30

if RUN_FULL_LLM_BENCHMARK:
    llm_eval_df = test_df.copy()
else:
    llm_eval_df = test_df.sample(
        min(LLM_SAMPLE_SIZE, len(test_df)),
        random_state=SEED,
    ).copy()

llm_eval_df = llm_eval_df.reset_index(drop=True)
print(f"LLM benchmark rows: {len(llm_eval_df)}")
display(llm_eval_df[["title", "entity_norm", "alert_flag", "model_text"]].head())


LLM benchmark rows: 126


,title,entity_norm,alert_flag,model_text
0,Хакеры четыре года грабили клиентов Сбербанка ...,ВТБ,1,Хакеры четыре года грабили клиентов Сбербанка ...
1,Порошенко ввел в действие решение СНБО о санкц...,Газпромбанк,0,Порошенко ввел в действие решение СНБО о санкц...
2,"Эксперты: технический дефолт по бондам ""Транса...",Альфа-Банк,0,"Эксперты: технический дефолт по бондам ""Транса..."
3,Крупные банки за месяц потеряли почти 10% вкладов,Райффайзенбанк,0,Крупные банки за месяц потеряли почти 10% вкла...
4,Red Wings выложила паспортные данные пассажиро...,ВТБ,1,Red Wings выложила паспортные данные пассажиро...


## 3. Prompt design

Prompt фиксируется до запуска benchmark. После просмотра результатов его не следует донастраивать на test, чтобы не подгонять результат под выборку.

Модель можно менять через переменную окружения `OPENAI_MODEL`. По умолчанию используется лёгкая модель для benchmark.


In [3]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    print(
        "OPENAI_API_KEY не найден. "
        "LLM benchmark является optional и будет пропущен."
    )
    OPENAI_AVAILABLE = False
else:
    OPENAI_AVAILABLE = True

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"OPENAI_MODEL: {OPENAI_MODEL}")


OPENAI_MODEL: gpt-4o-mini


In [4]:
SYSTEM_PROMPT = """
Ты — эксперт по банковским рискам.

Твоя задача — определить, содержит ли новостной фрагмент риск-сигнал для банка.

Риск-сигнал — это событие, которое может быть важно для мониторинга банковских рисков:
- кибератаки;
- утечки данных;
- мошенничество;
- сбои сервисов;
- блокировки;
- санкции;
- регуляторные претензии;
- ограничения деятельности;
- значимые операционные инциденты.

Если фрагмент содержит риск-сигнал, верни 1.
Если банк только упомянут, но значимого риск-события нет, верни 0.

Отвечай строго одной цифрой: 0 или 1.
Никаких пояснений.
"""


## 4. Few-shot examples

Few-shot примеры берутся только из train, чтобы не использовать объекты из valid/test в prompt.


In [5]:
if "train_df" not in globals():
    from pathlib import Path

    import numpy as np
    import pandas as pd

    SEED = globals().get("SEED", 42)
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "homework_04_dataset").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

    DATA_PATH = PROJECT_ROOT / "homework_04_dataset" / "data" / "dataset_for_training.csv"
    df = pd.read_csv(DATA_PATH)
    df["model_text"] = (
        df["title"].fillna("") + " " +
        df["text_fragment"].fillna("") + " " +
        df["entity_norm"].fillna("")
    )

    train_df = df[df["split"] == "train"].copy()
    valid_df = df[df["split"] == "valid"].copy()
    test_df = df[df["split"] == "test"].copy()

if "llm_eval_df" not in globals():
    if globals().get("RUN_FULL_LLM_BENCHMARK", True):
        llm_eval_df = test_df.copy()
    else:
        llm_eval_df = test_df.sample(
            min(globals().get("LLM_SAMPLE_SIZE", 30), len(test_df)),
            random_state=SEED,
        ).copy()
    llm_eval_df = llm_eval_df.reset_index(drop=True)

FEW_SHOT_EXAMPLES = []

few_shot_source = (
    train_df
    .groupby("alert_flag", group_keys=False)
    .apply(lambda x: x.sample(min(3, len(x)), random_state=SEED))
    .reset_index(drop=True)
)

for _, row in few_shot_source.iterrows():
    FEW_SHOT_EXAMPLES.append({
        "role": "user",
        "content": f"Фрагмент: {row['model_text'][:700]}",
    })
    FEW_SHOT_EXAMPLES.append({
        "role": "assistant",
        "content": str(int(row["alert_flag"])),
    })

print(f"Few-shot messages: {len(FEW_SHOT_EXAMPLES)}")
display(few_shot_source[["alert_flag", "title", "entity_norm"]])


Few-shot messages: 12


C:\Users\victo\AppData\Local\Temp\ipykernel_25644\3359370681.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(3, len(x)), random_state=SEED))


,alert_flag,title,entity_norm
0,0,"Молочный комплекс за 5,7 млрд рублей ввели в э...",Россельхозбанк
1,0,Новый способ вывода денег с банковских карт пр...,Т-Банк
2,0,Мошенники за год списали со счетов россиян до ...,Сбербанк
3,1,В Сбербанке произошел очередной сбой с банковс...,Сбербанк
4,1,У пользователей «Сбербанка Онлайн» на Android ...,Сбербанк
5,1,"Бывший топ-менеджер ""Сибнефти"" приговорен к 7 ...",ВТБ


## 5. Запуск zero-shot и few-shot classification

Результаты кешируются в `homework_05_06_modeling/reports/llm_benchmark`. Если файл predictions уже существует, notebook использует его и не делает повторные API-запросы.


In [6]:
import re
import time

LLM_REPORT_DIR = PROJECT_ROOT / "homework_05_06_modeling" / "reports" / "llm_benchmark"
LLM_REPORT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = LLM_REPORT_DIR / "llm_full_test_predictions.csv"
METRICS_PATH = LLM_REPORT_DIR / "llm_full_test_metrics.csv"


def parse_binary_answer(answer):
    answer = str(answer).strip()
    match = re.search(r"\b[01]\b", answer)
    return int(match.group(0)) if match else -1


try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
    OPENAI_AVAILABLE = False
    print("OpenAI SDK не установлен. LLM benchmark является optional и будет пропущен.")

if OPENAI_AVAILABLE and OpenAI is not None:
    client = OpenAI(api_key=OPENAI_API_KEY)
else:
    client = None


def classify_with_llm(text, few_shot=False, max_retries=3):
    if client is None:
        return -1

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if few_shot:
        messages.extend(FEW_SHOT_EXAMPLES)

    messages.append({
        "role": "user",
        "content": f"Фрагмент: {str(text)[:1000]}",
    })

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=messages,
                temperature=0,
                max_tokens=5,
            )
            answer = response.choices[0].message.content
            return parse_binary_answer(answer)

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                print(f"LLM request failed: {e}")
                return -1


In [7]:
use_cached_predictions = False

if PREDICTIONS_PATH.exists():
    cached_predictions_df = pd.read_csv(PREDICTIONS_PATH)
    prediction_cols = ["zero_shot_pred", "few_shot_pred"]
    cache_has_predictions = all(col in cached_predictions_df.columns for col in prediction_cols)
    cache_has_valid_answers = (
        cache_has_predictions and
        cached_predictions_df[prediction_cols].ne(-1).any().any()
    )

    if cache_has_valid_answers or not OPENAI_AVAILABLE:
        print(f"Загружаем кеш predictions: {PREDICTIONS_PATH}")
        llm_eval_df = cached_predictions_df
        use_cached_predictions = True
    else:
        print("Кеш predictions содержит только -1; пересчитываем LLM benchmark.")

if not use_cached_predictions:
    if OPENAI_AVAILABLE:
        zero_shot_preds = []

        for i, text in enumerate(llm_eval_df["model_text"].tolist()):
            pred = classify_with_llm(text, few_shot=False)
            zero_shot_preds.append(pred)

            if (i + 1) % 10 == 0:
                print(f"Zero-shot: {i + 1}/{len(llm_eval_df)} готово")

            time.sleep(0.3)

        few_shot_preds = []

        for i, text in enumerate(llm_eval_df["model_text"].tolist()):
            pred = classify_with_llm(text, few_shot=True)
            few_shot_preds.append(pred)

            if (i + 1) % 10 == 0:
                print(f"Few-shot: {i + 1}/{len(llm_eval_df)} готово")

            time.sleep(0.3)

        llm_eval_df["zero_shot_pred"] = zero_shot_preds
        llm_eval_df["few_shot_pred"] = few_shot_preds
        llm_eval_df.to_csv(PREDICTIONS_PATH, index=False)
        print(f"Predictions сохранены: {PREDICTIONS_PATH}")
    else:
        llm_eval_df["zero_shot_pred"] = -1
        llm_eval_df["few_shot_pred"] = -1
        print("LLM benchmark пропущен: OPENAI_API_KEY не задан или SDK недоступен.")

display(llm_eval_df[["alert_flag", "zero_shot_pred", "few_shot_pred", "title"]].head())


Кеш predictions содержит только -1; пересчитываем LLM benchmark.
Zero-shot: 10/126 готово
Zero-shot: 20/126 готово
Zero-shot: 30/126 готово
Zero-shot: 40/126 готово
Zero-shot: 50/126 готово
Zero-shot: 60/126 готово
Zero-shot: 70/126 готово
Zero-shot: 80/126 готово
Zero-shot: 90/126 готово
Zero-shot: 100/126 готово
Zero-shot: 110/126 готово
Zero-shot: 120/126 готово
Few-shot: 10/126 готово
Few-shot: 20/126 готово
Few-shot: 30/126 готово
Few-shot: 40/126 готово
Few-shot: 50/126 готово
Few-shot: 60/126 готово
Few-shot: 70/126 готово
Few-shot: 80/126 готово
Few-shot: 90/126 готово
Few-shot: 100/126 готово
Few-shot: 110/126 готово
Few-shot: 120/126 готово
Predictions сохранены: D:\6 - Обучение\ML Engineering\bank-news-risk-monitoring\homework_05_06_modeling\reports\llm_benchmark\llm_full_test_predictions.csv


,alert_flag,zero_shot_pred,few_shot_pred,title
0,1,1,1,Хакеры четыре года грабили клиентов Сбербанка ...
1,0,1,1,Порошенко ввел в действие решение СНБО о санкц...
2,0,1,1,"Эксперты: технический дефолт по бондам ""Транса..."
3,0,1,0,Крупные банки за месяц потеряли почти 10% вкладов
4,1,1,1,Red Wings выложила паспортные данные пассажиро...


## 6. Оценка качества LLM

Для LLM считаются только метрики классификации по классам: accuracy, precision, recall, F1, classification report и confusion matrix.

ROC-AUC и PR-AUC не считаются, потому что benchmark возвращает только классы без вероятностей.


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


def evaluate_llm_predictions(y_true, y_pred, model_name):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    valid_mask = y_pred != -1
    n_parse_errors = int((~valid_mask).sum())

    if valid_mask.sum() == 0:
        return {
            "model": model_name,
            "n_total": len(y_pred),
            "n_valid": 0,
            "n_parse_errors": n_parse_errors,
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1": None,
        }

    return {
        "model": model_name,
        "n_total": len(y_pred),
        "n_valid": int(valid_mask.sum()),
        "n_parse_errors": n_parse_errors,
        "accuracy": accuracy_score(y_true[valid_mask], y_pred[valid_mask]),
        "precision": precision_score(y_true[valid_mask], y_pred[valid_mask], zero_division=0),
        "recall": recall_score(y_true[valid_mask], y_pred[valid_mask], zero_division=0),
        "f1": f1_score(y_true[valid_mask], y_pred[valid_mask], zero_division=0),
    }


zero_shot_metrics = evaluate_llm_predictions(
    llm_eval_df["alert_flag"].to_numpy(),
    llm_eval_df["zero_shot_pred"].to_numpy(),
    "LLM zero-shot",
)
few_shot_metrics = evaluate_llm_predictions(
    llm_eval_df["alert_flag"].to_numpy(),
    llm_eval_df["few_shot_pred"].to_numpy(),
    "LLM few-shot",
)

llm_metrics_df = pd.DataFrame([zero_shot_metrics, few_shot_metrics])
llm_metrics_df.to_csv(METRICS_PATH, index=False)
print(f"Metrics сохранены: {METRICS_PATH}")
display(llm_metrics_df)


Metrics сохранены: D:\6 - Обучение\ML Engineering\bank-news-risk-monitoring\homework_05_06_modeling\reports\llm_benchmark\llm_full_test_metrics.csv


,model,n_total,n_valid,n_parse_errors,accuracy,precision,recall,f1
0,LLM zero-shot,126,126,0,0.730159,0.528571,0.973684,0.685185
1,LLM few-shot,126,126,0,0.674603,0.480000,0.947368,0.637168


In [13]:
for col, name in [
    ("zero_shot_pred", "LLM zero-shot"),
    ("few_shot_pred", "LLM few-shot"),
]:
    y_pred = llm_eval_df[col].to_numpy()
    valid_mask = y_pred != -1

    print(f"\n{name}: {valid_mask.sum()}/{len(y_pred)} успешно распознано")

    if valid_mask.sum() > 0:
        print(classification_report(
            llm_eval_df["alert_flag"].to_numpy()[valid_mask],
            y_pred[valid_mask],
            zero_division=0,
        ))

        print(confusion_matrix(
            llm_eval_df["alert_flag"].to_numpy()[valid_mask],
            y_pred[valid_mask],
        ))



LLM zero-shot: 126/126 успешно распознано
              precision    recall  f1-score   support

           0       0.98      0.62      0.76        88
           1       0.53      0.97      0.69        38

    accuracy                           0.73       126
   macro avg       0.76      0.80      0.72       126
weighted avg       0.85      0.73      0.74       126

[[55 33]
 [ 1 37]]

LLM few-shot: 126/126 успешно распознано
              precision    recall  f1-score   support

           0       0.96      0.56      0.71        88
           1       0.48      0.95      0.64        38

    accuracy                           0.67       126
   macro avg       0.72      0.75      0.67       126
weighted avg       0.82      0.67      0.68       126

[[49 39]
 [ 2 36]]


## 7. Анализ ошибок LLM

False negatives важны, потому что LLM пропускает реальные риск-сигналы. False positives важны, потому что они создают дополнительную нагрузку на аналитика.

Эти таблицы не означают автоматические ошибки разметки. Они показывают объекты, где LLM и исходная разметка расходятся. Такие строки полезны для экспертного review и улучшения annotation guideline.


In [14]:
base_error_columns = [
    "title",
    "text_fragment",
    "entity_norm",
    "source",
    "alert_flag",
    "risk_type_4cls",
]
base_error_columns = [col for col in base_error_columns if col in llm_eval_df.columns]

zero_shot_false_positives = llm_eval_df[
    llm_eval_df["alert_flag"].eq(0) & llm_eval_df["zero_shot_pred"].eq(1)
][base_error_columns + ["zero_shot_pred"]]

zero_shot_false_negatives = llm_eval_df[
    llm_eval_df["alert_flag"].eq(1) & llm_eval_df["zero_shot_pred"].eq(0)
][base_error_columns + ["zero_shot_pred"]]

print(f"Zero-shot false positives: {len(zero_shot_false_positives)}")
display(zero_shot_false_positives.head(10))

print(f"Zero-shot false negatives: {len(zero_shot_false_negatives)}")
display(zero_shot_false_negatives.head(10))


Zero-shot false positives: 33


,title,text_fragment,entity_norm,source,alert_flag,risk_type_4cls,zero_shot_pred
1,Порошенко ввел в действие решение СНБО о санкц...,Порошенко ввел в действие решение СНБО о санкц...,Газпромбанк,ods_tass,0,no_risk,1
2,"Эксперты: технический дефолт по бондам ""Транса...","Эксперты: технический дефолт по бондам ""Транса...",Альфа-Банк,buriy,0,no_risk,1
3,Крупные банки за месяц потеряли почти 10% вкладов,Крупные банки за месяц потеряли почти 10% вкла...,Райффайзенбанк,taiga_fontanka,0,no_risk,1
6,Сбербанк опровергает проблемы с обслуживанием ...,Сбербанк опровергает проблемы с обслуживанием ...,Сбербанк,taiga_fontanka,0,no_risk,1
9,Максим Сушинский: Постоянно получаю предложени...,Максим Сушинский: Постоянно получаю предложени...,ВТБ,buriy,0,no_risk,1
19,"На Android появился вирус, который ворует день...","На Android появился вирус, который ворует день...",Росбанк,telegram_contest,0,no_risk,1
22,Крупные банки за месяц потеряли почти 10% вкладов,Крупные банки за месяц потеряли почти 10% вкла...,Альфа-Банк,taiga_fontanka,0,no_risk,1
30,ЦБ: влияние падения цен на нефть и санкций на ...,ЦБ: влияние падения цен на нефть и санкций на ...,ВТБ,ods_tass,0,no_risk,1
31,«Аэрофлот» подал второй иск к «Трансаэро» еще ...,«Аэрофлот» подал второй иск к «Трансаэро» еще ...,Альфа-Банк,buriy,0,no_risk,1
32,"Канада ввела санкции против Сечина, Костина и ...","Канада ввела санкции против Сечина, Костина и ...",ВТБ,ods_tass,0,no_risk,1


Zero-shot false negatives: 1


,title,text_fragment,entity_norm,source,alert_flag,risk_type_4cls,zero_shot_pred
24,Глава Сбербанка отчитался о «косяках»,Глава Сбербанка отчитался о «косяках» Он подче...,Сбербанк,lenta,1,operational_risk,0


In [11]:
few_shot_false_positives = llm_eval_df[
    llm_eval_df["alert_flag"].eq(0) & llm_eval_df["few_shot_pred"].eq(1)
][base_error_columns + ["few_shot_pred"]]

few_shot_false_negatives = llm_eval_df[
    llm_eval_df["alert_flag"].eq(1) & llm_eval_df["few_shot_pred"].eq(0)
][base_error_columns + ["few_shot_pred"]]

print(f"Few-shot false positives: {len(few_shot_false_positives)}")
display(few_shot_false_positives.head(10))

print(f"Few-shot false negatives: {len(few_shot_false_negatives)}")
display(few_shot_false_negatives.head(10))


Few-shot false positives: 39


,title,text_fragment,entity_norm,source,alert_flag,risk_type_4cls,few_shot_pred
1,Порошенко ввел в действие решение СНБО о санкц...,Порошенко ввел в действие решение СНБО о санкц...,Газпромбанк,ods_tass,0,no_risk,1
2,"Эксперты: технический дефолт по бондам ""Транса...","Эксперты: технический дефолт по бондам ""Транса...",Альфа-Банк,buriy,0,no_risk,1
9,Максим Сушинский: Постоянно получаю предложени...,Максим Сушинский: Постоянно получаю предложени...,ВТБ,buriy,0,no_risk,1
19,"На Android появился вирус, который ворует день...","На Android появился вирус, который ворует день...",Росбанк,telegram_contest,0,no_risk,1
21,Следствие вернулось к делу о хищении из бюджет...,Следствие вернулось к делу о хищении из бюджет...,ВТБ,lenta,0,no_risk,1
22,Крупные банки за месяц потеряли почти 10% вкладов,Крупные банки за месяц потеряли почти 10% вкла...,Альфа-Банк,taiga_fontanka,0,no_risk,1
23,Обанкротился производитель водки на березовых ...,Обанкротился производитель водки на березовых ...,Альфа-Банк,taiga_fontanka,0,no_risk,1
28,"Чья «Северная верфь», того и гендиректор","Чья «Северная верфь», того и гендиректор Мы сч...",Альфа-Банк,taiga_fontanka,0,no_risk,1
30,ЦБ: влияние падения цен на нефть и санкций на ...,ЦБ: влияние падения цен на нефть и санкций на ...,ВТБ,ods_tass,0,no_risk,1
31,«Аэрофлот» подал второй иск к «Трансаэро» еще ...,«Аэрофлот» подал второй иск к «Трансаэро» еще ...,Альфа-Банк,buriy,0,no_risk,1


Few-shot false negatives: 2


,title,text_fragment,entity_norm,source,alert_flag,risk_type_4cls,few_shot_pred
24,Глава Сбербанка отчитался о «косяках»,Глава Сбербанка отчитался о «косяках» Он подче...,Сбербанк,lenta,1,operational_risk,0
25,Сбербанк на два часа запретит операции через и...,Сбербанк на два часа запретит операции через и...,Сбербанк,lenta,1,operational_risk,0


## 8. Ограничения benchmark

- Benchmark зависит от внешнего API, доступности ключа и версии LLM.
- Результат зависит от prompt design.
- Prompt не донастраивается на test после просмотра результатов.
- Запросы имеют стоимость, поэтому при необходимости можно переключиться на подвыборку test.
- Benchmark не используется для обучения или выбора моделей в основном проекте.
- ROC-AUC и PR-AUC не считаются, потому что LLM возвращает только класс без вероятности.


## Итоговый вывод

LLM zero-shot/few-shot benchmark на полном test показал высокий recall по положительному классу: `0.97` для zero-shot и `0.95` для few-shot. Это означает, что LLM почти не пропускает фрагменты, которые в датасете размечены как риск-сигналы.

При этом precision заметно ниже: `0.53` для zero-shot и `0.48` для few-shot. Следовательно, LLM склонна к over-alerting: она часто относит спорные или пограничные фрагменты к риск-сигналам, создавая много false positives.

С точки зрения оценки разметки это полезный результат. Высокий recall показывает, что большинство положительных примеров в разметке согласуются с внешней LLM-оценкой: LLM также считает их риск-сигналами. Это косвенно подтверждает, что положительный класс `alert_flag = 1` в целом размечен содержательно и не выглядит случайным.

Основная зона для ручной проверки — false positives LLM, то есть случаи, где LLM ставит `1`, а в датасете стоит `alert_flag = 0`. Такие расхождения не следует автоматически считать ошибками разметки: LLM может быть слишком осторожной и расширительно трактовать риск. Однако эти объекты являются хорошими кандидатами для повторной экспертной проверки, потому что часть из них может оказаться пограничными риск-сигналами.

False negatives LLM встречаются редко: `1` случай для zero-shot и `2` случая для few-shot. Эти строки также стоит проверить вручную, так как они могут указывать либо на сложные формулировки риска, либо на неоднозначность исходной разметки.

Zero-shot оказался лучше few-shot на текущем prompt и выбранных примерах. Это показывает, что few-shot не гарантирует автоматический прирост качества: примеры нужно подбирать отдельно, а не просто добавлять случайные объекты из train.

### Ориентир для дальнейшего обучения моделей

LLM benchmark показывает желательный профиль качества для задачи риск-мониторинга: модель должна стремиться к высокому recall по положительному классу, потому что пропуск реального риск-сигнала является более критичной ошибкой, чем дополнительная ручная проверка ложного алерта.

Однако одного высокого recall недостаточно. Результаты LLM показывают, что при слишком широком понимании риска появляется много false positives. Поэтому при дальнейшем обучении моделей целевой профиль качества должен быть следующим:

1. сохранить высокий recall по классу `alert_flag = 1`;
2. повысить precision относительно LLM benchmark;
3. снизить количество false positives без резкого роста false negatives;
4. использовать F1 как баланс между полнотой обнаружения риск-сигналов и нагрузкой на аналитика;
5. отдельно анализировать ошибки на пограничных и спорных новостных фрагментах.

Таким образом, LLM benchmark можно использовать как внешний ориентир: он показывает, что задачу можно решать в recall-oriented режиме, но также демонстрирует главный риск такого подхода — избыточное число ложных алертов. Поэтому хорошая обучаемая модель должна не просто повторять LLM, а лучше контролировать баланс между recall и precision.

Итоговая оценка разметки: разметка выглядит в целом пригодной для обучения и анализа, особенно по положительному классу. При этом есть зона неопределённости среди отрицательных и пограничных примеров, которую стоит улучшать через ручную перепроверку false positives/false negatives LLM, уточнение инструкции разметки и возможное добавление второго разметчика.

LLM benchmark следует рассматривать как вспомогательный инструмент контроля качества разметки и поиска спорных объектов, а не как источник истины. Его результаты зависят от prompt, версии LLM, внешнего API, стоимости запросов и доступности ключа.

### Практическое использование результатов LLM benchmark

По результатам LLM benchmark можно сформировать список объектов для ручной проверки:

1. `LLM false positives` — LLM считает фрагмент риск-сигналом, а в датасете стоит `alert_flag = 0`.
2. `LLM false negatives` — LLM не видит риск, а в датасете стоит `alert_flag = 1`.
3. Пограничные случаи, где zero-shot и few-shot дают разные ответы.

Такая проверка может повысить качество следующей версии датасета, уточнить правила разметки и задать ориентиры для дальнейшего обучения моделей.
